# PDF dokumentu analīze ar Python (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/blob/main/pdf_tasks/student_workbook_pdf_analize_lv_colab.ipynb)

Šī ir **Google Colab draudzīga** un pilnībā izpildāma studentu workbook versija tēmai **"PDF dokumentu analīze ar Python"**.

Atšķirībā no lokālās studentu versijas, šajā notebook:

- visas nepieciešamās bibliotēkas tiek uzstādītas Colab vidē;
- tiek uzstādīti arī OCR rīki `Tesseract` un `Poppler`;
- tiek automātiski lejupielādēti sagatavotie faili no repozitorija mapes `pdf_tasks`;
- koda šūnas ir pilnībā aizpildītas un izpildāmas no sākuma līdz beigām.

## Ievads

Šī notebook mērķis ir soli pa solim parādīt, kā ar Python analizēt dažādu tipu PDF dokumentus:

- digitāli ģenerētus PDF ar teksta slāni;
- PDF failus ar tabulām;
- skenētus PDF dokumentus, kuriem vajadzīga OCR pieeja;
- vairāku failu batch apstrādi vienā darba plūsmā.

Praktiski mēs mēģinām sasniegt vienu galveno mērķi: izvēlēties pareizo rīku pareizajam PDF tipam un iegūt rezultātu, ko pēc tam var saglabāt kā tekstu, tabulu vai kopsavilkuma atskaiti.

Šajā tēmā lielākā daļa grūtību parasti nav pašā Python sintaksē, bet gan tehniskajā sagatavošanā:

- pareizo bibliotēku uzstādīšanā;
- ārējo rīku, piemēram, `Tesseract` un `Poppler`, pieejamībā;
- atšķirībā starp digitālu PDF un skenētu PDF;
- datu kvalitātes problēmās pēc ekstraktēšanas vai OCR.

Google Colab šeit palīdz, jo lielu daļu no šīs tehniskās sagatavošanas var automatizēt ar dažām uzstādīšanas šūnām notebook sākumā.


## Darba plūsma Google Colab vidē

Ieteicams izpildīt šūnas tieši šādā secībā:

1. uzstādīt sistēmas rīkus OCR un PDF apstrādei;
2. uzstādīt Python bibliotēkas;
3. lejupielādēt `pdf_tasks` piemērfailus no GitHub;
4. importēt bibliotēkas un ielādēt datus;
5. iziet cauri sadaļām par tekstu, tabulām, OCR un batch apstrādi.

Piezīme:

- Colab faili glabājas tikai sesijas laikā;
- ja vēlaties saglabāt rezultātus ārpus Colab, lejupielādējiet tos sesijas beigās.


## 0. Sistēmas rīku uzstādīšana Colab vidē

OCR un PDF lapu renderēšanai Colab nepietiek tikai ar Python bibliotēkām.

Šajā šūnā ar termināļa komandām tiks uzstādīti:

- `tesseract-ocr` OCR dzinējs;
- `tesseract-ocr-lav` latviešu valodas atbalsts;
- `poppler-utils`, kas vajadzīgs `pdf2image`.


In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-lav poppler-utils
# ! Jupyter Notebook vidē izpilda sistēmas terminaļa komandas kas būtu bez !

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-lav.
Preparing to unpack .../tesseract-ocr-lav_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-lav (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-lav (1:4.00~git30-7274cfa-1.1) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!dir

sample_data


## 1. Python bibliotēku uzstādīšana Colab vidē

Šajā šūnā tiks uzstādītas visas notebook izmantotās Python bibliotēkas:

- `PyPDF2`
- `pdfplumber`
- `pandas`
- `pytesseract`
- `pdf2image`
- `Pillow`

Ja Colab paziņo par nepieciešamu runtime restartu, palaidiet pārējās šūnas vēlreiz pēc restartēšanas.


In [3]:
!pip install -q --disable-pip-version-check PyPDF2 pdfplumber pandas pytesseract pdf2image Pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 67.7 MB/s eta 0:00:00


In [4]:
# pārbaudīsim kādas mums ir versijas katram rīkam
import PyPDF2 # biblioteka kļust pieejam mūsu darbam
print(f"PyPDF2 versija {PyPDF2.__version__}") # daudzas nopietnas bibliotekas piedāva __version__ mainīgo kas parāda

PyPDF2 versijs 3.0.1


In [5]:
# interesanti bet PyPDF2 ir agriezies atpakaļ uz PyPDF sākot no PyPDF 3.1.0
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 15.6 MB/s eta 0:00:00


In [6]:
# apskatīsim kādas ir pydpf versija
import pypdf
print(f"pypdf versija {pypdf.__version__}")
# tātad pēdejos 3-4 gadus ir daudz jaunumu, visticam nebūs ar vienkāršiem PDF failiem saistīti

pypdf versija 6.10.2


## 2. Sagatavoto `pdf_tasks` failu lejupielāde no repozitorija

Lai notebook darbotos pilnībā arī Colab vidē, šī šūna lejupielādēs visus vajadzīgos failus no GitHub:

- galvenos PDF piemērus;
- to salīdzināšanas/reference failus;
- batch piemērus;
- manifestu ar aprakstu par datu kopu.


In [7]:
!mkdir -p pdf_tasks/batch pdf_tasks/outputs
# iepriekšējā rinda izveido mapi ar apakšmapēm ar mkdir komandu
# nākošais ir piemērs kā no kāda tīmekļā resursa - šeit manis izveidotā github repozitorija
# bet varētu būt jebkurš publisks fails
# var lejuplādēt failus
from pathlib import Path
from urllib.request import urlretrieve

# izveidojam string mainīgo ar bāzes adresi
# lielie burti simbolizē konstantes - bet negarantē
BASE_RAW = "https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks"

# tad Python sarakstā glabājam failu nosaukumus - stringus
ASSET_PATHS = [
    "pdf_dataset_manifest.json",
    "batch_expected_summary.csv",
    "digital_text.pdf",
    "digital_text_source.txt",
    "table_report.pdf",
    "table_report_source.csv",
    "scanned_document.pdf",
    "scanned_document_page1.png",
    "scanned_document_reference.txt",
    "batch/digital_brief.pdf",
    "batch/attendance_report.pdf",
    "batch/scan_archive_copy.pdf",
    "batch/scan_archive_copy_page1.png",
]

# izvēlamies sakni kur likt failus
root = Path("pdf_tasks")

# ar for ciklu iesim cauri individuāliem saraksta elementiem (failu nosaukumiem)
for rel_path in ASSET_PATHS:
    # tātad rel_path saturēs vienu individuāli elementu no ASSET_PATH satura
    # katram failam izvedosim pilno adresi
    target = root / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    url = f"{BASE_RAW}/{rel_path}"
    print(f"Lejuplādējam no {url}")
    urlretrieve(url, target) # mēs izsaucām tātad šo Python standarta bibliotekas funkciju urretrieve
    print("Downloaded:", rel_path)


Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/pdf_dataset_manifest.json
Downloaded: pdf_dataset_manifest.json
Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/batch_expected_summary.csv
Downloaded: batch_expected_summary.csv
Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/digital_text.pdf
Downloaded: digital_text.pdf
Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/digital_text_source.txt
Downloaded: digital_text_source.txt
Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/table_report.pdf
Downloaded: table_report.pdf
Lejuplādējam no https://raw.githubusercontent.com/ValRCS/RTU_Digitalas_Prasmes_Excel_VBA_Python/main/pdf_tasks/table_report_source.csv
Downloaded: table_report_sou

## 3. Importi un vides pārbaude

Šajā šūnā:

- importēsim bibliotēkas;
- pārbaudīsim, vai notebook tiešām darbojas Google Colab vidē;
- izdrukāsim daļu no uzstādīto rīku versijām.


In [8]:
from __future__ import annotations

# parasti liekam Python standarta bibliotekas vispirms
import json
import re
import subprocess
import sys
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from datetime import datetime
print(f"Patlaban ir {datetime.now()}")

import pandas as pd # Lielā datu analīzes biblioteka, kurai standarta alias ir pd
import pdfplumber
import pytesseract
# ar from BIBLIOTEK import apaksmodulis/funkcija/klase  mēs tikai kaut ko konkrētu imporējam
# to daram kad visu nevajag
from PIL import Image
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
from IPython.display import Markdown, display

# šī būs pārbaude vai patiešām esam google Colab vidē
# ja neesam (teiksim lokali) tad iespējams var kaut kas nestrādāt
IN_COLAB = "google.colab" in sys.modules

print("Running in Colab:", IN_COLAB)
print("Python version:", sys.version.split()[0])
print("Tesseract path:", subprocess.run(["which", "tesseract"], capture_output=True, text=True).stdout.strip())
print("pdftoppm path:", subprocess.run(["which", "pdftoppm"], capture_output=True, text=True).stdout.strip())


Patlaban ir 2026-04-22 06:46:46.246326
Running in Colab: True
Python version: 3.12.13
Tesseract path: /usr/bin/tesseract
pdftoppm path: /usr/bin/pdftoppm


## 4. Ceļi un palīgfunkcijas

Šeit definējam:

- ceļus uz `pdf_tasks` failiem;
- teksta tīrīšanas funkcijas;
- PDF lapu skaita iegūšanu;
- teksta ekstraktēšanu;
- tabulu ekstraktēšanu;
- OCR funkciju skenētiem PDF.


In [9]:
# tagad vairāk uzstādam tādus nekritiskus mainīgus un opcijas
# piemēram nākoša rinda uzstāda cik kolonnas rādit maksimums
pd.set_option("display.max_colwidth", 120)

# savukārt nākošie ir mainīgie kur dzīvos mūsu faili
PDF_ROOT = Path("pdf_tasks")
BATCH_DIR = PDF_ROOT / "batch"
OUTPUT_DIR = PDF_ROOT / "outputs"
# izdrukāsim kur kas dzīvos
print(f"PDF_ROOT {PDF_ROOT}")
print(f"BATCH_DIR {BATCH_DIR}")
print(f"OUTPUT_DIR {OUTPUT_DIR}")

MANIFEST_PATH = PDF_ROOT / "pdf_dataset_manifest.json"
BATCH_EXPECTED_PATH = PDF_ROOT / "batch_expected_summary.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(text: str) -> str:
    if not text:
        return ""
    # šeit mēs rakstītu kādu teksta apstrādi veikt normalizācijā
    # tātad šeir ir gan vienkāŗs replace
    # gan arī regular expression bāzets replace
    # liekat šeit paši savus vai visus atkomentējiet ja nevajag
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text) # aizstājam 3+ jaunrindas ar tieši 2
    return text.strip()


def load_manifest() -> dict:
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def get_primary_asset(kind: str) -> Path:
    manifest = load_manifest()
    for item in manifest["primary_inputs"]:
        if item["kind"] == kind:
            return PDF_ROOT / item["path"]
    raise KeyError(f"Primary asset not found for kind={kind!r}")


def get_primary_entry(kind: str) -> dict:
    manifest = load_manifest()
    for item in manifest["primary_inputs"]:
        if item["kind"] == kind:
            return item
    raise KeyError(f"Primary asset not found for kind={kind!r}")


def pdf_page_count(pdf_path: Path) -> int:
    return len(PdfReader(str(pdf_path)).pages)


def extract_text_with_pypdf2(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path)) # izveido jaunu reader objektu
    # izveido sarakstu(list) ar lapu saturu - daļām
    # tehniska piezīme izmantojam list comprehension
    parts = [(page.extract_text() or "") for page in reader.pages]
    # visbeidzot visu salipinam kopa divām jaunām rindam STARP lapām
    return normalize_text("\n\n".join(parts))


def extract_text_with_pdfplumber(pdf_path: Path) -> str:
    with pdfplumber.open(pdf_path) as pdf:
        parts = [(page.extract_text() or "") for page in pdf.pages]
    return normalize_text("\n\n".join(parts))


def extract_first_table(pdf_path: Path) -> pd.DataFrame:
    with pdfplumber.open(pdf_path) as pdf:
        table = pdf.pages[0].extract_table()

    if not table:
        return pd.DataFrame()

    header = [str(col).strip() if col is not None else "" for col in table[0]]
    rows = table[1:]
    df = pd.DataFrame(rows, columns=header)
    df = df.dropna(how="all")

    for column in df.columns:
        df[column] = df[column].map(lambda value: str(value).strip() if value is not None else "")

    return df


def guess_pdf_type(file_name: str) -> str:
    name = file_name.lower()
    if "scan" in name or "ocr" in name:
        return "scanned"
    if "table" in name or "report" in name or "attendance" in name:
        return "table-heavy"
    return "digital-text"


def ocr_pdf_first_page(pdf_path: Path, fallback_image: Path | None = None, lang: str = "eng+lav") -> tuple[Image.Image, str]:
    try:
        images = convert_from_path(str(pdf_path), first_page=1, last_page=1)
        image = images[0]
    except Exception:
        if fallback_image is None or not fallback_image.exists():
            raise
        image = Image.open(fallback_image)

    text = pytesseract.image_to_string(image, lang=lang)
    return image, normalize_text(text)


def process_batch_pdf(pdf_path: Path) -> dict:
    pdf_type = guess_pdf_type(pdf_path.name)
    page_count = pdf_page_count(pdf_path)

    pypdf2_text = extract_text_with_pypdf2(pdf_path)
    pdfplumber_text = extract_text_with_pdfplumber(pdf_path)
    best_text = pypdf2_text if len(pypdf2_text) >= len(pdfplumber_text) else pdfplumber_text

    table_df = extract_first_table(pdf_path) if pdf_type == "table-heavy" else pd.DataFrame()

    status = "ok"
    notes = ""
    if pdf_type == "scanned":
        status = "ocr-needed"
        notes = "PyPDF2 and pdfplumber see little or no text. OCR is the correct next step."

    return {
        "file_name": pdf_path.name,
        "pdf_type_guess": pdf_type,
        "page_count": page_count,
        "text_chars": len(best_text),
        "table_rows": int(len(table_df)) if not table_df.empty else 0,
        "status": status,
        "notes": notes,
    }


manifest = load_manifest()
manifest


PDF_ROOT pdf_tasks
BATCH_DIR pdf_tasks/batch
OUTPUT_DIR pdf_tasks/outputs


{'root': 'D:\\Github\\RTU_Digitalas_Prasmes_Excel_VBA_Python\\pdf_tasks',
 'primary_inputs': [{'path': 'digital_text.pdf',
   'kind': 'digital-text',
   'page_count': 3,
   'companion': 'digital_text_source.txt'},
  {'path': 'table_report.pdf',
   'kind': 'table-heavy',
   'page_count': 1,
   'companion': 'table_report_source.csv'},
  {'path': 'scanned_document.pdf',
   'kind': 'scanned',
   'page_count': 1,
   'companion': 'scanned_document_reference.txt',
   'fallback_image': 'scanned_document_page1.png'}],
 'batch_inputs': [{'file_name': 'digital_brief.pdf',
   'pdf_type_guess': 'digital-text',
   'page_count': 2,
   'expected_text_chars_min': 350,
   'expected_table_rows_min': 0,
   'notes': 'Text extraction should work with PyPDF2.'},
  {'file_name': 'attendance_report.pdf',
   'pdf_type_guess': 'table-heavy',
   'page_count': 1,
   'expected_text_chars_min': 50,
   'expected_table_rows_min': 4,
   'notes': 'First page should contain an extractable table.'},
  {'file_name': 'scan_

## 5. Pieejamie faili

Apskatīsim, kādi faili tika lejupielādēti no `pdf_tasks`.


In [10]:
print("Primary inputs:")
for item in manifest["primary_inputs"]:
    print("-", item["path"], "| kind:", item["kind"], "| pages:", item["page_count"])

print("\nBatch inputs:")
for item in manifest["batch_inputs"]:
    print("-", item["file_name"], "| kind:", item["pdf_type_guess"], "| pages:", item["page_count"])

print("\nManifest notes:")
for note in manifest["notes"]:
    print("-", note)


Primary inputs:
- digital_text.pdf | kind: digital-text | pages: 3
- table_report.pdf | kind: table-heavy | pages: 1
- scanned_document.pdf | kind: scanned | pages: 1

Batch inputs:
- digital_brief.pdf | kind: digital-text | pages: 2
- attendance_report.pdf | kind: table-heavy | pages: 1
- scan_archive_copy.pdf | kind: scanned | pages: 1

Manifest notes:
- The notebook reads main sample PDFs directly from pdf_tasks.
- Batch examples live in pdf_tasks/batch.
- The scanned PNG is a fallback for OCR workflows on machines without Poppler.


# 1. ak. st. PDF faila atvēršana un struktūras izpēte

Šajā akadēmiskajā stundā studenti iepazīstas ar pašu PDF dokumentu kā datu avotu. Mērķis nav vēl veikt dziļu analīzi, bet saprast, kā Python redz dokumentu:

- vai dokumentam ir vairākas lapas;
- vai dokumentam ir teksta slānis;
- kādu metadatu informāciju var nolasīt;
- cik daudz teksta katrā lapā vispār ir pieejams.

**Izmantotie rīki**

- `PyPDF2.PdfReader` dokumenta atvēršanai;
- `reader.pages` lapu sarakstam;
- `reader.metadata` dokumenta metadatiem;
- `page.extract_text()` sākotnējam teksta pārbaudījumam.

**Ko darīs koda šūnas**

- atvērs `digital_text.pdf`;
- parādīs lapu skaitu un metadatus;
- izies cauri katrai lapai;
- parādīs, cik rakstzīmju katrā lapā iespējams iegūt.


In [12]:
digital_pdf = get_primary_asset("digital-text")
# izdrukāsim ko mēs atvērsim
print(f"Atvērsim PDF failu: {digital_pdf}")
# varējam vienkārši digital_pdf vertību uzstādīt ar roku - ierakstīt ka string
# tātad mums ir PdfReader importēts no PyPDF2 bibliotekas
digital_reader = PdfReader(str(digital_pdf))
# mēs iegustam šo digital_reader objektu kurš satur visu par mūsu pdf failu

print("Fails:", digital_pdf.name) # šis nāk no Python Path objekta, ja digital_pdf būtu string tad nestrādāu
print("Lapu skaits:", len(digital_reader.pages))
print("Metadati:")
# izdurkāsim visus metadatus par šo failu ar ciklu jo digital_reader.metadata ir Python vardnica
for key, value in digital_reader.metadata.items():
    print(f"- {key}: {value}")

# ejam caur visām lapām ar ciklu
for page_number, page in enumerate(digital_reader.pages, start=1):
    # iegustam
    page_text = page.extract_text() or ""
    print(f"Lapa {page_number}: teksta garums = {len(page_text)}")


Atvērsim PDF failu: pdf_tasks/digital_text.pdf
Fails: digital_text.pdf
Lapu skaits: 3
Metadati:
- /Author: OpenAI Codex
- /CreationDate: D:20260421222419+03'00'
- /Creator: (unspecified)
- /Keywords: 
- /ModDate: D:20260421222419+03'00'
- /Producer: ReportLab PDF Library - (opensource)
- /Subject: (unspecified)
- /Title: Digital workbook example
- /Trapped: /False
Lapa 1: teksta garums = 344
Lapa 2: teksta garums = 348
Lapa 3: teksta garums = 267


In [14]:
# pamēģināsim to pašu drusku vienkāršāk un pa soļiem
# izmantosim jaunāko versiju no https://pypi.org/project/pypdf/
# mēs jau to importējam bet parādīšu velreiz
import pypdf # ja importē vēlreiz parasti importi ir saglabāti un tas nav nekas slikts bet protams lieki importet nav vajadzība tas jādara tikai vienreiz viena piezimju grāmata vai skriptā
print(f"pypdf versija {pypdf.__version__}")
# lai nebūtu tā sauktais "name collission" ar pypdf2 PdfReader funkciju
# šoreiz es izmantošu pilnu pierakstu
my_pdf_reader = pypdf.PdfReader("pdf_tasks/digital_text.pdf")
# šajā brīdi ja nav kļūdas man viss ir pieejams par šo pdf failu šajā piezimju grāmata ar šo my_pdf_reader mainīgo
print("Lapu skaits:", len(my_pdf_reader.pages))


pypdf versija 6.10.2
Lapu skaits: 3


## Refleksija un dokumentācija pēc 1. ak. st.

**Refleksijas jautājumi**

- Kā pēc `extract_text()` rezultāta var spriest, vai PDF ir digitāli ģenerēts?
- Kāda ir atšķirība starp lapu skaitu, metadatiem un pašu satura ekstraktēšanu?
- Kurā brīdī būtu jāsāk domāt par OCR, nevis par parastu PDF teksta nolasīšanu?

**Autoritatīvi avoti tālākai lasīšanai**

- [PyPDF2 dokumentācija: sākumlapa](https://pypdf2.readthedocs.io/en/3.x/)
- [PyPDF2 dokumentācija: metadata](https://pypdf2.readthedocs.io/en/3.0.0/user/metadata.html)
- [pypdf dokumentācija: PdfReader klase](https://pypdf.readthedocs.io/en/stable/modules/PdfReader.html)


# 2. ak. st. Teksta ekstraktēšana ar PyPDF2

Šajā akadēmiskajā stundā fokuss ir uz teksta ieguvi no digitāli ģenerēta PDF. Šis ir tipisks scenārijs, kad dokumentā jau eksistē teksta slānis un OCR nav vajadzīgs.

**Izmantotie rīki**

- `page.extract_text()` vienas lapas tekstam;
- iepriekš definētā `extract_text_with_pypdf2()` funkcija visa dokumenta tekstam;
- `normalize_text()` vienkāršai pēcapstrādei;
- `Counter` biežāko vārdu skaitīšanai;
- `pandas.DataFrame` rezultātu pārskatāmam attēlojumam.

**Ko darīs koda šūnas**

- parādīs pirmās lapas teksta priekšskatījumu;
- savāks tekstu no visām lapām vienā blokā;
- iztīrīs liekās atstarpes un tukšās rindas;
- saglabās rezultātu `.txt` failā;
- salīdzinās iegūto tekstu ar reference tekstu;
- aprēķinās biežāk sastopamos vārdus.


In [16]:
# atgādinu ka digital_reader ir šis PdfReader objekts
print("digital_reader datu tips", type(digital_reader))

digital_reader datu tips <class 'PyPDF2._reader.PdfReader'>


In [18]:
first_page_text = digital_reader.pages[0].extract_text() or "" # šis ir tā sauktais ternary operators
# mēs varējam parbaudīt ar if extract_text() atgriež None un tad piešķirt šo saturu
# apskatam pirmos 1000 symbolus
print(first_page_text[:1000])


Digital workbook example
This PDF is designed for PyPDF2 text extraction exercises.
Students can inspect page counts, metadata, and paragraph text without needing OCR.
The document mixes short headings, complete sentences, and repeated keywords such as data,
report, and student.
Use this file when the notebook asks for a digital text sample.



In [19]:
# apskatam pilno sauturu bez print - tehniski nevis __str__ bet __repr__ kas domāts programmētājiem
first_page_text

'Digital workbook example\nThis PDF is designed for PyPDF2 text extraction exercises.\nStudents can inspect page counts, metadata, and paragraph text without needing OCR.\nThe document mixes short headings, complete sentences, and repeated keywords such as data,\nreport, and student.\nUse this file when the notebook asks for a digital text sample.\n'

In [20]:
clean_text = extract_text_with_pypdf2(digital_pdf)
text_output_path = OUTPUT_DIR / "sample_pdf_text_colab.txt"
text_output_path.write_text(clean_text, encoding="utf-8")

source_text = normalize_text((PDF_ROOT / "digital_text_source.txt").read_text(encoding="utf-8"))
similarity_ratio = SequenceMatcher(None, clean_text.lower(), source_text.lower()).ratio()

print("Saglabāts:", text_output_path)
print("Iegūtā teksta garums:", len(clean_text))
print("Reference teksta garums:", len(source_text))
print("Līdzības koeficients:", round(similarity_ratio, 4))

display(Markdown("### Teksta priekšskatījums"))
print(clean_text[:1500])


Saglabāts: pdf_tasks/outputs/sample_pdf_text_colab.txt
Iegūtā teksta garums: 960
Reference teksta garums: 960
Līdzības koeficients: 0.999


### Teksta priekšskatījums

Digital workbook example
This PDF is designed for PyPDF2 text extraction exercises.
Students can inspect page counts, metadata, and paragraph text without needing OCR.
The document mixes short headings, complete sentences, and repeated keywords such as data,
report, and student.
Use this file when the notebook asks for a digital text sample.

Lesson notes
Page two contains enough text to test multi-page extraction and simple cleaning steps.
Typical tasks include joining pages, counting words, and saving the result to a text file.
Because the text layer is present, extraction should work directly with PyPDF2.
Students can compare this result with the OCR example later in the workbook.

Reflection prompts
Which library worked best for this document type?
How much cleanup was needed after extraction?
Which repeated words describe the document most clearly?
These questions are intentionally simple so the sample stays predictable for automated checks.


In [21]:
words = re.findall(r"[A-Za-zĀ-ž0-9]{3,}", clean_text.lower())
top_words = pd.DataFrame(Counter(words).most_common(15), columns=["word", "count"]) # tikai 15
# ja grib visus vai citus tad 15 jānomaina
top_words # top_words ir Dataframe to var arī saglabāt kā CSV un arī Excel un citos formātus


,word,count
0,the,8
1,text,6
2,and,5
3,this,4
4,for,4
5,extraction,4
6,page,3
7,document,3
8,digital,2
9,workbook,2


In [22]:
# augšminētajā šunā mēs sadalījam individuālajos vārdos un saskaitījam biežumu
words[:10]

['digital',
 'workbook',
 'example',
 'this',
 'pdf',
 'designed',
 'for',
 'pypdf2',
 'text',
 'extraction']

In [23]:
# saglabāsim top_words kā CSV, likšu galvenā mapē (nav pilnīgi pareiza vieta)
top_words.to_csv("top_words.csv", index=False, encoding="utf-8")

#

In [25]:
# ja esam Google Colab tad varam arī lejuplādet automatiski
# we need files module from google colab to download
from google.colab import files # files ir tikai Google Colab
files.download("top_words.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Refleksija un dokumentācija pēc 2. ak. st.

**Refleksijas jautājumi**

- Vai ekstraktētais teksts pilnībā sakrita ar reference tekstu?
- Kādas kļūdas parasti parādās PDF teksta ekstraktēšanā: rindkopas, atstarpes, secība, galvenes?
- Kāpēc teksta pēcapstrāde ir gandrīz vienmēr vajadzīga arī digitāliem PDF?

**Autoritatīvi avoti tālākai lasīšanai**

- [pypdf dokumentācija: Extract Text from a PDF](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)
- [PyPDF2 dokumentācija: sākumlapa un user guide](https://pypdf2.readthedocs.io/en/3.x/)
- [pandas dokumentācija: DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


## 2b - lapu izvilkšanu un savienošana jaunā PDF failā

In [29]:
# So we have a PDF file and we would like generate a new PDF file with just the first and last page from this PDF as file_name_first_last.pdf
file_path = Path("pdf_tasks/digital_text.pdf") # atkal Path ir Python failu ceļu palīgbiblioteka
file_name = file_path.name # tātad digital_text.pdf ja ar roku
print(f"Full Path {file_path}")
# assert that full path exists - jo savādak nav vērts no gultas celties :)
assert file_path.exists(), f"Nav šada faila šāda vietā {file_path}"
# ja assert nav kļūda tav varam droši turpinam, ja kļūda Te apstāsimies
print(f"File Name {file_name}")

Full Path pdf_tasks/digital_text.pdf
File Name digital_text.pdf


In [31]:
# let's open the file_path with PdfReader
pdf_file_object = PdfReader(file_path)
# how many pages?
print(f"Lapu skaits {len(pdf_file_object.pages)}")
#


Lapu skaits 3


In [32]:
# apskatamies ko mēs varam pateikt par pirmo lapu?
print(pdf_file_object.pages[0]) # redzam dažādu lapas iekšpuses informāciju, kuru pārsvarā negribēsim šodien skatīt

{'/Contents': IndirectObject(10, 0, 137052460891792), '/MediaBox': [0, 0, 595.2756, 841.8898], '/Parent': IndirectObject(9, 0, 137052460891792), '/Resources': {'/Font': IndirectObject(1, 0, 137052460891792), '/ProcSet': ['/PDF', '/Text', '/ImageB', '/ImageC', '/ImageI']}, '/Rotate': 0, '/Trans': {}, '/Type': '/Page'}


In [34]:
# let's make a new pdf object with only the first and last page
needed_pages = [pdf_file_object.pages[0], pdf_file_object.pages[-1]] # 0 ir pirmā un -1 ir pēdejā
# ja ir tikai viena lapa tad būs divas identiskas lapas :)
# let's write back to first_last.pdf these two pages

from PyPDF2 import PdfWriter # mums vajag rakstītāju

writer = PdfWriter() #izveidojam jaun rakstītāju
# let's add the creator name "Valdis"
writer.add_metadata({"/Creator": "Valdis"})
# how about place or creation - "Rīga"
writer.add_metadata({"/Producer": "RīgasTehniskā Universitāte"})
# ja nebūs / tad vai nu kļūda vai citi PDF lasītāji var nesaprast šos metadatus

# ejam cauri savai kolekcijai un pievienojam lapas
# te varētu būt lapas arī no vairākiem pdf savāktas
for page in needed_pages:
    writer.add_page(page)

output_filename = OUTPUT_DIR / "first_last.pdf"
with open(output_filename, "wb") as out_pdf:
    writer.write(out_pdf)

print(f"Saved first and last pages to: {output_filename}")

Saved first and last pages to: pdf_tasks/outputs/first_last.pdf


# 3. ak. st. Tabulu ekstraktēšana ar `pdfplumber`

Šajā akadēmiskajā stundā uzsvars ir uz strukturētu datu ieguvi no PDF. Tabulas ir sarežģītākas nekā vienkāršs teksts, jo PDF formātā tabula bieži ir tikai vizuāli izkārtots saturs.

**Izmantotie rīki**

- `pdfplumber.open()` PDF lapu objektu apstrādei;
- `page.extract_table()` lielākās tabulas iegūšanai;
- `pandas.DataFrame` tabulas strukturēšanai;
- `DataFrame.to_csv()` eksportam.

**Ko darīs koda šūnas**

- atvērs `table_report.pdf`;
- mēģinās iegūt pirmo tabulu no pirmās lapas;
- pārvērtīs tabulas datus par `DataFrame`;
- saglabās rezultātu CSV formātā;
- salīdzinās iegūto tabulu ar oriģinālo `table_report_source.csv`.


In [ ]:
table_pdf = get_primary_asset("table-heavy")
extracted_table_df = extract_first_table(table_pdf)
extracted_table_df


In [ ]:
table_output_path = OUTPUT_DIR / "table_extract_colab.csv"
extracted_table_df.to_csv(table_output_path, index=False, encoding="utf-8")

expected_table_df = pd.read_csv(PDF_ROOT / "table_report_source.csv")
extracted_compare_df = extracted_table_df.astype(str).reset_index(drop=True)
expected_compare_df = expected_table_df.astype(str).reset_index(drop=True)
exact_match = extracted_compare_df.equals(expected_compare_df)

print("Saglabāts:", table_output_path)
print("Rindu skaits:", len(extracted_table_df))
print("Kolonnu skaits:", len(extracted_table_df.columns))
print("Sakrīt ar reference CSV:", exact_match)

display(Markdown("### Reference CSV"))
display(expected_table_df)


## Refleksija un dokumentācija pēc 3. ak. st.

**Refleksijas jautājumi**

- Vai tabula tika atrasta automātiski bez papildu parametriem?
- Kādas problēmas būtu iespējamas reālos PDF: sapludinātas šūnas, tukšas kolonnas, nobīdīti virsraksti?
- Kad ar `extract_table()` pietiek, un kad būtu vajadzīga dziļāka tabulas debugēšana?

**Autoritatīvi avoti tālākai lasīšanai**

- [pdfplumber PyPI apraksts](https://pypi.org/project/pdfplumber/)
- [pdfplumber GitHub dokumentācija un piemēri](https://github.com/jsvine/pdfplumber)
- [pandas dokumentācija: DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


# 4. ak. st. OCR tehnoloģijas skenētiem PDF

Šajā akadēmiskajā stundā tiek aplūkota situācija, kurā PDF satur tikai attēlu vai skenētu lapu. Šādā gadījumā parasta teksta ekstraktēšana dod maz vai neko, un jāizmanto OCR.

**Izmantotie rīki**

- `pdf2image.convert_from_path()` PDF lapas pārvēršanai attēlā;
- `PIL.Image` attēla glabāšanai un attēlošanai;
- `pytesseract.image_to_string()` teksta atpazīšanai;
- `normalize_text()` OCR rezultāta tīrīšanai.

**Ko darīs koda šūnas**

- atvērs `scanned_document.pdf`;
- pārvērtīs pirmo lapu par attēlu;
- palaidīs OCR ar valodām `eng+lav`;
- saglabās OCR rezultātu teksta failā;
- salīdzinās OCR rezultātu ar reference tekstu.


In [ ]:
scanned_entry = get_primary_entry("scanned")
scanned_pdf = PDF_ROOT / scanned_entry["path"]
scanned_fallback = PDF_ROOT / scanned_entry["fallback_image"]

scanned_image, clean_ocr_text = ocr_pdf_first_page(scanned_pdf, fallback_image=scanned_fallback, lang="eng+lav")

print("Fails:", scanned_pdf.name)
print("OCR teksta garums:", len(clean_ocr_text))
display(scanned_image)


In [ ]:
ocr_output_path = OUTPUT_DIR / "scanned_document_ocr_colab.txt"
ocr_output_path.write_text(clean_ocr_text, encoding="utf-8")

scanned_reference = normalize_text((PDF_ROOT / "scanned_document_reference.txt").read_text(encoding="utf-8"))
ocr_similarity = SequenceMatcher(None, clean_ocr_text.lower(), scanned_reference.lower()).ratio()

print("Saglabāts:", ocr_output_path)
print("Reference garums:", len(scanned_reference))
print("OCR līdzība pret reference tekstu:", round(ocr_similarity, 4))

display(Markdown("### OCR rezultāts"))
print(clean_ocr_text)

display(Markdown("### Reference teksts"))
print(scanned_reference)


## Refleksija un dokumentācija pēc 4. ak. st.

**Refleksijas jautājumi**

- Cik labi OCR rezultāts sakrita ar reference tekstu?
- Kā OCR kvalitāti ietekmē skenējuma kvalitāte, kontrasts un valodas modelis?
- Kāpēc OCR ir atsevišķs solis un nevis vienkārši "tas pats, tikai PDF formātā"?

**Autoritatīvi avoti tālākai lasīšanai**

- [pytesseract PyPI dokumentācija](https://pypi.org/project/pytesseract/)
- [Tesseract User Manual](https://tesseract-ocr.github.io/tessdoc/)
- [pdf2image PyPI dokumentācija](https://pypi.org/project/pdf2image/)


# 5. ak. st. Batch apstrāde vairākiem PDF failiem

Šajā akadēmiskajā stundā tiek savienots viss iepriekš apgūtais vienā automatizētā darba plūsmā. Mērķis ir vienā ciklā apstrādāt vairākus PDF failus ar dažādu tipu saturu.

**Izmantotie rīki**

- `Path.glob()` failu atlasīšanai mapē;
- `guess_pdf_type()` dokumenta tipa minēšanai;
- `process_batch_pdf()` vienotas apstrādes funkcijai;
- `pandas.DataFrame` kopsavilkuma tabulai;
- `DataFrame.merge()` rezultātu salīdzināšanai ar reference aprakstu.

**Ko darīs koda šūnas**

- atradīs visus PDF failus mapē `pdf_tasks/batch`;
- katram failam noteiks tipu un lapu skaitu;
- mēģinās iegūt tekstu vai tabulas informāciju;
- saglabās rezultātu kopsavilkuma CSV failā;
- salīdzinās iegūtos rezultātus ar sagaidāmo `batch_expected_summary.csv`.


In [ ]:
batch_files = sorted(BATCH_DIR.glob("*.pdf"))
summary_rows = [process_batch_pdf(path) for path in batch_files]
summary_df = pd.DataFrame(summary_rows)

batch_output_path = OUTPUT_DIR / "batch_summary_colab.csv"
summary_df.to_csv(batch_output_path, index=False, encoding="utf-8")

print("Saglabāts:", batch_output_path)
summary_df


In [ ]:
expected_batch_df = pd.read_csv(BATCH_EXPECTED_PATH)

comparison_df = summary_df.merge(
    expected_batch_df,
    on=["file_name", "pdf_type_guess"],
    how="left",
    suffixes=("_actual", "_expected"),
)

comparison_df["page_count_ok"] = comparison_df["page_count_actual"] == comparison_df["page_count_expected"]
comparison_df["text_min_ok"] = comparison_df["text_chars"] >= comparison_df["expected_text_chars_min"]
comparison_df["table_min_ok"] = comparison_df["table_rows"] >= comparison_df["expected_table_rows_min"]

comparison_df[
    [
        "file_name",
        "pdf_type_guess",
        "page_count_actual",
        "page_count_expected",
        "text_chars",
        "expected_text_chars_min",
        "table_rows",
        "expected_table_rows_min",
        "page_count_ok",
        "text_min_ok",
        "table_min_ok",
    ]
]


## Refleksija un dokumentācija pēc 5. ak. st.

**Refleksijas jautājumi**

- Kā batch apstrāde palīdzētu reālā biroja vai datu analīzes darbā?
- Kādi riski parādās, ja mapē ir dažādu tipu PDF faili?
- Kā jūs paplašinātu šo pipeline, lai tā automātiski izvēlētos arī OCR gadījumus?

**Autoritatīvi avoti tālākai lasīšanai**

- [Python 3.14 dokumentācija: pathlib](https://docs.python.org/3/library/pathlib.html)
- [pandas user guide: Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas API: merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)


# 6. Mini projekts ar automātisku pieejas izvēli

Lai notebook būtu pilnībā izpildāms, šajā daļā mini projekts tiek realizēts automātiski:

- izvēlamies vienu PDF failu;
- nosakām piemērotāko pieeju;
- saglabājam rezultātu;
- izdrukājam īsu kopsavilkumu.


In [ ]:
def run_mini_project(pdf_path: Path) -> dict:
    pdf_type = guess_pdf_type(pdf_path.name)

    if pdf_type == "digital-text":
        output_path = OUTPUT_DIR / f"{pdf_path.stem}_mini_text.txt"
        result_text = extract_text_with_pypdf2(pdf_path)
        output_path.write_text(result_text, encoding="utf-8")
        return {
            "file_name": pdf_path.name,
            "method": "text",
            "output_file": str(output_path),
            "status": "done",
            "chars": len(result_text),
        }

    if pdf_type == "table-heavy":
        output_path = OUTPUT_DIR / f"{pdf_path.stem}_mini_table.csv"
        result_df = extract_first_table(pdf_path)
        result_df.to_csv(output_path, index=False, encoding="utf-8")
        return {
            "file_name": pdf_path.name,
            "method": "table",
            "output_file": str(output_path),
            "status": "done",
            "rows": len(result_df),
        }

    fallback = PDF_ROOT / "scanned_document_page1.png"
    _, result_text = ocr_pdf_first_page(pdf_path, fallback_image=fallback, lang="eng+lav")
    output_path = OUTPUT_DIR / f"{pdf_path.stem}_mini_ocr.txt"
    output_path.write_text(result_text, encoding="utf-8")
    return {
        "file_name": pdf_path.name,
        "method": "ocr",
        "output_file": str(output_path),
        "status": "done",
        "chars": len(result_text),
    }


project_pdf = get_primary_asset("scanned")
project_result = run_mini_project(project_pdf)
project_result


## Iegūtie rezultāti

Šajā sesijā tika izveidoti rezultātu faili mapē `pdf_tasks/outputs`.

Tie parasti ietver:

- ekstraktēto tekstu no digitāla PDF;
- tabulas CSV failu;
- OCR rezultātu;
- batch kopsavilkumu;
- mini projekta rezultātu.


In [ ]:
output_files = sorted(OUTPUT_DIR.glob("*"))
pd.DataFrame(
    [{"file_name": path.name, "size_bytes": path.stat().st_size} for path in output_files]
)


## Papildu piezīme par Colab

Colab darba vide ir īslaicīga. Ja vēlaties saglabāt izveidotos rezultātu failus lokāli, varat tos lejupielādēt.


In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Lai lejupielādētu kādu failu, izmantojiet:")
    print("files.download('pdf_tasks/outputs/sample_pdf_text_colab.txt')")
    print("files.download('pdf_tasks/outputs/table_extract_colab.csv')")
    print("files.download('pdf_tasks/outputs/scanned_document_ocr_colab.txt')")
    print("files.download('pdf_tasks/outputs/batch_summary_colab.csv')")
else:
    print("Šī lejupielādes palīdzība ir paredzēta Google Colab videi.")


## Noslēguma secinājumi

Pēc šī notebook izpildes var secināt:

- **digitāliem PDF** labi strādā `PyPDF2`;
- **tabulu PDF** gadījumā noderīgs ir `pdfplumber`;
- **skenētiem dokumentiem** vajadzīga OCR pieeja ar `pytesseract`;
- **batch apstrāde** ļauj vienā plūsmā apstrādāt vairākus dažādu tipu dokumentus.


## Plašāka informācija un atsauces

Ja vēlaties šo tēmu apgūt dziļāk, zemāk ir pārbaudīti un autoritatīvi resursi, kuros var lasīt vairāk par konkrētajiem rīkiem un pieejām.

### PDF lasīšana, struktūra un teksta ekstraktēšana

- [PyPDF2 dokumentācija](https://pypdf2.readthedocs.io/en/3.x/)
- [pypdf dokumentācija: Extract Text from a PDF](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)
- [pypdf dokumentācija: PdfReader klase](https://pypdf.readthedocs.io/en/stable/modules/PdfReader.html)

### Tabulu ekstraktēšana no PDF

- [pdfplumber oficiālais projekts GitHub](https://github.com/jsvine/pdfplumber)
- [pdfplumber PyPI lapa](https://pypi.org/project/pdfplumber/)

### OCR un skenētu dokumentu apstrāde

- [pytesseract PyPI lapa](https://pypi.org/project/pytesseract/)
- [Tesseract User Manual](https://tesseract-ocr.github.io/tessdoc/)
- [pdf2image PyPI lapa](https://pypi.org/project/pdf2image/)

### Darbs ar datiem un rezultātu saglabāšana

- [pandas dokumentācija: DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [pandas dokumentācija: merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [pandas user guide: Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)

### Failu ceļi un Python pamatbibliotēka

- [Python 3 dokumentācija: pathlib](https://docs.python.org/3/library/pathlib.html)

### Ko ieteicams lasīt vispirms

Ja mērķis ir ātri sākt praktisku darbu:

1. izlasiet `pypdf` vai `PyPDF2` teksta ekstraktēšanas sadaļu;
2. pēc tam apskatiet `pdfplumber` tabulu ekstraktēšanu;
3. tad izlasiet `Tesseract` un `pytesseract` dokumentāciju OCR sadaļai;
4. beigās nostipriniet rezultātu saglabāšanu ar `pandas`.
